# 5.05 — Albert quantification of SRR3997504 (per-chromosome L1 aggregation)

Port of legacy `4.02-ach-albert_quantify_SRR3997504` into the R/ggplot reproducibility
scheme. **Stage A** produces
`results/srr3997504/aggregate_by_chr.csv` (`sample, figure, chrom, method, class, value`).
**Stage B** (this notebook) renders the four by-chromosome figures to
`reports/figures/srr3997504/`:

- **`SRR3997504_agreggate_mlvs_bowtie_by_chr`** — ML (Albert scores) vs Bowtie2 /
  RepeatMasker, unfiltered.
- **`SRR3997504_filtered_agreggate_mlvs_star_by_chr`** — ML (P(NEGATIVE) < 0.3) vs STAR of
  filtered reads vs STAR of all reads.
- **`SRR3997504_filtered_agreggate_l1em_mlvs_star_by_chr`** — adds L1EM (L1EM.400 intervals
  on the filtered STAR alignment).
- **`SRR3997504_filtered_agreggate_l1em_bwa_by_chr`** — ML vs L1EM re-aligned with BWA.

Classes: L1HS / L1PA2 / L1PA / LINE/L1. Combined "Method (Class)" legend inside the panel;
same data, axes, methods, and ordering as `notebooks/images/SRR3997504_*_by_chr.png`.


In [ ]:
## Reproducibility & environment capture -------------------------------
set.seed(3469)
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(jsonlite)
})
options(repr.plot.width = 12, repr.plot.height = 7)
theme_set(theme_bw(base_size = 28.125))


In [ ]:
## Paths, provenance, palette ------------------------------------------
results_dir <- file.path("..", "results", "srr3997504")
figures_dir <- file.path("..", "reports", "figures", "srr3997504")
dir.create(figures_dir, recursive = TRUE, showWarnings = FALSE)
cat("results_dir:", results_dir, "\n")
cat("figures_dir:", figures_dir, "\n")

manifest_path <- file.path(results_dir, "manifest.json")
if (file.exists(manifest_path)) {
  manifest <- fromJSON(manifest_path)
  cat("\n-- compute manifest --\n")
  cat("git_commit:", manifest$git_commit, "\n")
  cat("timestamp: ", manifest$timestamp_utc, "\n")
  cat("seed:      ", manifest$seed, "\n")
  if (!is.null(manifest$seed)) stopifnot(manifest$seed == 3469)
} else {
  warning("manifest.json not found")
}

# The legacy figures encode subfamily CLASS by colour and quantification METHOD by fill
# pattern (solid / hatched / open), with a single combined "Method (Class)" legend inside the
# panel. Reproduce that with one fill series per (method, class): the class fixes the hue
# (fixed palette; red #D62728 reserved) and the method lightens it (primary method darkest).
class_palette <- c("L1HS" = "#1F77B4", "L1PA2" = "#FF7F0E", "L1PA" = "#2CA02C",
                   "LINE/L1" = "#9467BD", "NEGATIVE" = "#7F7F7F")
class_levels  <- names(class_palette)
chrom_levels  <- c(paste0("chr", 1:22), "chrX", "chrY", "chrMT", "chrM")
source(file.path("..", "scripts/R/trap_plot_helpers.R"))
save_fig <- trap_save_fig(figures_dir, default_width = 12, default_height = 7)


In [ ]:
## Load the tidy table + validation checks ----------------------------
agg_path <- file.path(results_dir, "aggregate_by_chr.csv")
stopifnot("aggregate_by_chr.csv not found — run the Stage-A translator" = file.exists(agg_path))
agg <- fread(agg_path)

req <- c("sample", "figure", "chrom", "method", "class", "value")
stopifnot("missing required columns" = all(req %in% names(agg)))
stopifnot("value must be finite and >= 0" = all(is.finite(agg$value) & agg$value >= 0))

std_chrom <- c(paste0("chr", 1:22), "chrX", "chrY", "chrMT", "chrM")
bad_chrom <- setdiff(unique(agg$chrom), std_chrom)
if (length(bad_chrom) > 0)
  warning(sprintf("non-standard chrom(s) present: %s", paste(bad_chrom, collapse = ", ")))

cat("Loaded", nrow(agg), "rows; figures:\n")
print(agg[, .N, by = figure])


In [ ]:
## Grouped-bar helper (per chromosome; combined Method (Class) legend) --
# One figure = one `figure` value. Bars are dodged per chromosome, ordered class-major /
# method-minor (legacy column order); the legend is method-major (legacy column order).
# fill = "Method (Class)" (class hue + method lightness). `legend`: "inside" (top-left, boxed —
# the 4.01-sourced 5.03 figures) or an outside keyword like "right" (the 4.02-sourced 5.04
# figures, whose combined legend is too wide to sit inside). Missing combos -> no bar.


In [ ]:
## ML (Albert) vs Bowtie2/RepeatMasker (unfiltered)
plot_by_chr(agg, "SRR3997504_agreggate_mlvs_bowtie_by_chr", method_levels = c("ML", "Bowtie2"), legend = "right", width = 14)


In [ ]:
## ML (filtered) vs STAR filtered vs STAR whole
plot_by_chr(agg, "SRR3997504_filtered_agreggate_mlvs_star_by_chr", method_levels = c("ML", "FILTERED_STAR", "WHOLE_STAR"), legend = "right", width = 14)


In [ ]:
## + L1EM (L1EM.400 intervals on filtered STAR)
plot_by_chr(agg, "SRR3997504_filtered_agreggate_l1em_mlvs_star_by_chr", method_levels = c("ML", "FILTERED_STAR", "WHOLE_STAR", "L1EM_STAR"), legend = "right", legend_size = 14, width = 14)


In [ ]:
## ML vs L1EM (BWA re-alignment), aggregated by L1EM locus chromosome
plot_by_chr(agg, "SRR3997504_filtered_agreggate_l1em_bwa_by_chr", method_levels = c("ML", "L1EM_BWA"), legend = "right", width = 14)
